In [ ]:
!nvidia-smi
!python -V
!nvcc --version || true

In [ ]:
%%bash
set -e

# Conda'yı aktif et
source /opt/conda/etc/profile.d/conda.sh

# Eski kurulum kalıntılarını temizle
conda env remove -n nndet -y || true
rm -rf /kaggle/working/nnDetection

# Python 3.8 ortamı oluştur
conda create -y -n nndet python=3.8
conda activate nndet

# Temel araçlar
conda install -y -c conda-forge git pip cmake ninja
conda install -y -c conda-forge gxx_linux-64==9.3.0

# CUDA 11.3 + PyTorch 1.11
conda install -y -c nvidia/label/cuda-11.3.1 cuda
conda install -y -c pytorch pytorch==1.11.0 torchvision==0.12.0 torchaudio==0.11.0 cudatoolkit=11.3

# Repo
cd /kaggle/working
git clone https://github.com/MIC-DKFZ/nnDetection.git
cd nnDetection

# Derleyici yolları
export CXX=$CONDA_PREFIX/bin/x86_64-conda_cos6-linux-gnu-c++
export CC=$CONDA_PREFIX/bin/x86_64-conda_cos6-linux-gnu-cc

# Kaggle GPU mimarisini otomatik almaya çalış
export TORCH_CUDA_ARCH_LIST=$(python - <<'PY'
import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"{major}.{minor}")
else:
    print("7.5")
PY
)

# Ortam değişkenleri
export det_data=/kaggle/working/nnDetData
export det_models=/kaggle/working/nnDetModels
export OMP_NUM_THREADS=1
export det_num_threads=2
export det_verbose=1

mkdir -p $det_data $det_models

# Eski bağımlılıklarla uyum için
pip install --upgrade pip setuptools wheel
pip install "numpy<1.24" "protobuf<4"

# nnDetection bağımlılıkları
pip install \
  "pytorch_lightning>=1.3.1,<=1.4.2" \
  "batchgenerators>=0.24" \
  "nnunet==1.7.1" \
  scipy scikit-learn "scikit-image>=0.14" pandas \
  "PyYAML>=5.1,!=5.4.*" \
  nevergrad dicom2nifti medpy "SimpleITK<2.1.0" \
  tqdm loguru "hydra-core>=1.1.0" mlflow GitPython \
  matplotlib seaborn "torchmetrics>=0.7.0,<=0.7.3"

pip install hydra-core --upgrade --pre
pip install git+https://github.com/mibaumgartner/pytorch_model_summary.git

# CUDA extension derlemesi
FORCE_CUDA=1 pip install -v -e .

# Kurulum testi
python - <<'PY'
import torch
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import nndet._C
import nndet
print("nnDetection kurulumu başarılı.")
PY